# Chapter 15 — Memory Search as a Random Walk

Companion notebook for the [probintro textbook](https://josephausterweil.github.io/probintro/intro2/15_memory_search/).

Abbott, Austerweil & Griffiths (2012): human semantic fluency is a **censored random walk** on a semantic network. We build a small network, sample walks, apply the censoring function, compute IRTs, and watch the position-1-slowest signature emerge — with no switch rule.

## Setup

In [ ]:
!pip install genjax -q
print('✅ ready')

## The network

Two animal clusters (pets, big animals) joined by a bridge (tiger), plus non-animal distractors and a cue node.

In [ ]:
import jax.numpy as jnp
import numpy as np

names  = ["dog", "cat", "hamster", "tiger", "lion", "zebra", "giraffe",
          "house", "food", "water", "animal"]
is_animal = np.array([1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0])
word_len  = np.array([3, 3, 7, 5, 4, 5, 7, 5, 4, 5, 6])
idx = {n: i for i, n in enumerate(names)}
N = len(names)

edges = [
    ("dog", "cat"), ("dog", "hamster"), ("cat", "hamster"),
    ("lion", "zebra"), ("lion", "giraffe"), ("zebra", "giraffe"),
    ("cat", "tiger"), ("tiger", "lion"),
    ("dog", "house"), ("house", "food"), ("food", "water"),
    ("water", "cat"), ("hamster", "food"),
    ("animal", "dog"), ("animal", "lion"),
]
L = np.zeros((N, N))
for a, b in edges:
    L[idx[a], idx[b]] = 1.0
    L[idx[b], idx[a]] = 1.0
L = jnp.array(L)

degree = L.sum(axis=1)
P = L / degree[:, None]
print("network:", N, "nodes,", int(L.sum() // 2), "edges")

## Sample many walks (batched)

In [ ]:
import jax
import jax.random as jr

LOGP = jnp.log(jnp.where(P > 0, P, 1e-30))
STEPS = 400

def one_walk(key):
    def step(node, k):
        nxt = jr.categorical(k, LOGP[node])
        return nxt, nxt
    _, visited = jax.lax.scan(step, idx["animal"], jr.split(key, STEPS))
    return visited

walk_batch = jax.jit(jax.vmap(one_walk))
walks = np.array(walk_batch(jr.split(jr.key(0), 500)))
print("sampled", walks.shape[0], "walks of", walks.shape[1], "steps each")

## Censor, compute IRTs, find the signature

Keep each animal's first visit; IRT = wander-time between first-hits + word length; label by position within the patch. Position 1 (first word of a new cluster) should be slowest.

In [ ]:
from collections import defaultdict

cluster = {"dog": 0, "cat": 0, "hamster": 0,
           "lion": 1, "zebra": 1, "giraffe": 1, "tiger": 2}

def censored_irts(walk):
    seen, taus, words = set(), [], []
    for t, node in enumerate(walk):
        if is_animal[node] and node not in seen:
            seen.add(node); taus.append(t); words.append(node)
    irts = [taus[k] - taus[k-1] + int(word_len[words[k]])
            for k in range(1, len(words))]
    return words, irts

def patch_positions(words):
    cl = [cluster[names[w]] for w in words]
    pos, run = [], 1
    for k in range(1, len(words)):
        run = 1 if (cl[k] != cl[k-1] and 2 not in (cl[k], cl[k-1])) else run + 1
        pos.append(run)
    return pos

by_position = defaultdict(list)
for walk in walks:
    words, irts = censored_irts(walk)
    if len(irts) < 3:
        continue
    for p, irt in zip(patch_positions(words), irts):
        if p <= 3:
            by_position[p].append(irt)

avg = np.mean([x for v in by_position.values() for x in v])
print(f"average IRT over all positions: {avg:.1f}")
for p in [1, 2, 3]:
    ratio = np.mean(by_position[p]) / avg
    print(f"patch position {p}: IRT / average = {ratio:.2f}")

## Inverting the walk: estimating the network from behaviour

Running the model *backward* — recover the network (or its structure) from fluency lists — is hard because the censored list is a deterministic function of the latent path with the path marginalized away. Conditioning a `@gen` model on the list directly is degenerate (almost every sampled path disagrees, so weights collapse). That is why the published estimator, **U-INVITE** (Zemla & Austerweil, 2018), computes the censored-walk likelihood *analytically* with the **fundamental matrix** of an absorbing chain, rebuilding the absorbing/transient split after each reported animal.

If we only want a *coarse* feature of the network, we can sidestep the analytic likelihood and let **simulation** do the work. Assume K known clusters, fully connected within, with within-prob >> between-prob; the contrast r = p_out/p_in is the one number that matters.

In [ ]:
import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np

K, PER = 3, 3
N = K * PER
cluster_of = np.repeat(np.arange(K), PER)

def block_transition(p_in, p_out):
    same_cluster = cluster_of[:, None] == cluster_of[None, :]
    W = np.where(same_cluster, p_in, p_out)
    np.fill_diagonal(W, 0.0)
    return jnp.array(W / W.sum(axis=1, keepdims=True), dtype=jnp.float32)

def walk(key, start, steps, LOGP):
    def step(s, k):
        nxt = jr.categorical(k, LOGP[s]); return nxt, nxt
    _, vis = jax.lax.scan(step, start, jr.split(key, steps))
    return jnp.concatenate([jnp.array([start]), vis])

def censor(vis):
    seen, rep = set(), []
    for n in np.array(vis):
        n = int(n)
        if n not in seen:
            seen.add(n); rep.append(n)
    return rep

def clustering_score(rep):
    if len(rep) < 2:
        return 0.0
    return np.mean([cluster_of[rep[i]] == cluster_of[rep[i+1]]
                    for i in range(len(rep) - 1)])

Simulation-based (ABC) inference over the contrast r: weight candidate r values by how close their simulated clustering score is to the observed one.

In [ ]:
def mean_score(key, r, n_lists=8, steps=80):
    LOGP = jnp.log(block_transition(1.0, r))
    keys = jr.split(key, n_lists)
    walks = jax.vmap(lambda k: walk(k, 0, steps, LOGP))(keys)
    return float(np.mean([clustering_score(censor(w)) for w in np.array(walks)]))

R_GRID = np.array([0.1, 0.2, 0.4, 0.7, 1.0])

def abc_posterior(observed_score, key, bandwidth=0.05):
    weights = []
    for i, r in enumerate(R_GRID):
        sims = np.array([mean_score(jr.fold_in(key, i * 100 + j), float(r))
                         for j in range(6)])
        weights.append(np.exp(-((sims - observed_score) ** 2) / (2 * bandwidth ** 2)).mean())
    w = np.array(weights)
    return w / w.sum()

strong_obs = mean_score(jr.key(0), 0.1)
weak_obs   = mean_score(jr.key(0), 0.7)
print(f"strong-cluster data: clustering score {strong_obs:.2f}")
print(f"weak-cluster   data: clustering score {weak_obs:.2f}")

post_strong = abc_posterior(strong_obs, jr.key(1))
post_weak   = abc_posterior(weak_obs,   jr.key(1))
print("posterior over r (grid 0.1, 0.2, 0.4, 0.7, 1.0):")
print("  strong data ->", np.round(post_strong, 2), " MAP r =", R_GRID[post_strong.argmax()])
print("  weak data   ->", np.round(post_weak, 2),   " MAP r =", R_GRID[post_weak.argmax()])

## Try it yourself

1. Delete the bridge edges (`cat–tiger`, `tiger–lion`) so the clusters disconnect. Re-run — can one walk now report from both clusters? What happens to the position-1 signature?
2. Add more distractor nodes between the clusters (a longer bridge path). Does the position-1 IRT ratio go up or down? Relate to the τ(k) − τ(k−1) term.